In [3]:
import pandas as pd

example = pd.DataFrame({
    "price" : [500, 502, 501, 505, 507, 504,510]
})

In [5]:
# Price 3 trading days in the future
example["price_t+3"] = example["price"].shift(-3)

print(example["price_t+3"])

0    505.0
1    507.0
2    504.0
3    510.0
4      NaN
5      NaN
6      NaN
Name: price_t+3, dtype: float64


In [8]:
# Future 3-day return

example["future_return_3d"] = (
    example["price_t+3"] / example["price"] - 1
)

print(example["future_return_3d"])

0    0.010000
1    0.009960
2    0.005988
3    0.009901
4         NaN
5         NaN
6         NaN
Name: future_return_3d, dtype: float64


In [10]:
# Binary target:
# 1 = future return > 0.5%
# 0 = otherwise

example["target_3d"] = (
    example["future_return_3d"] > 0.005
).astype(int)

example

,price,price_t+3,future_return_3d,target_3d
0,500,505.0,0.010000,1
1,502,507.0,0.009960,1
2,501,504.0,0.005988,1
3,505,510.0,0.009901,1
4,507,NaN,NaN,0
5,504,NaN,NaN,0
6,510,NaN,NaN,0


In [13]:
# shift(-3) -> takes the row 3 row up 
# shift(-3) -> gelecek (for making target)
# shift(3) -> gecmis (for making feature)
#cause in our dataset dates gets close to present when it goes down

In [15]:
## Data Leakage and Look-Ahead Bias

model_df = example.dropna(
    subset=["future_return_3d"]
).copy()

model_df["target_3d"] = (
    model_df["future_return_3d"] > 0.005
).astype(int)

model_df

,price,price_t+3,future_return_3d,target_3d
0,500,505.0,0.010000,1
1,502,507.0,0.009960,1
2,501,504.0,0.005988,1
3,505,510.0,0.009901,1


In [16]:
example["past_price_t-3"] = example["price"].shift(3)
example["future_price_t+3"] = example["price"].shift(-3)

example[
    ["price", "past_price_t-3", "future_price_t+3"]
]

,price,past_price_t-3,future_price_t+3
0,500,NaN,505.0
1,502,NaN,507.0
2,501,NaN,504.0
3,505,500.0,510.0
4,507,502.0,NaN
5,504,501.0,NaN
6,510,505.0,NaN


In [19]:
features = ["price"]

X = model_df[features]
y = model_df["target_3d"]

print("Features:")
display(X)

print("Target:")
display(y)


Features:


,price
0,500
1,502
2,501
3,505


Target:


0    1
1    1
2    1
3    1
Name: target_3d, dtype: int64

In [20]:
split_idx = int(len(model_df) * 0.80)

train = model_df.iloc[:split_idx].copy()
test = model_df.iloc[split_idx:].copy()

print("TRAIN")
display(train)

print("TEST")
display(test)

TRAIN


,price,price_t+3,future_return_3d,target_3d
0,500,505.0,0.010000,1
1,502,507.0,0.009960,1
2,501,504.0,0.005988,1


TEST


,price,price_t+3,future_return_3d,target_3d
3,505,510.0,0.009901,1


In [21]:
features = ["price"]

X_train = train[features]
y_train = train["target_3d"]

X_test = test[features]
y_test = test["target_3d"]

print("X_train")
display(X_train)

print("y_train")
display(y_train)

print("X_test")
display(X_test)

print("y_test")
display(y_test)

X_train


,price
0,500
1,502
2,501


y_train


0    1
1    1
2    1
Name: target_3d, dtype: int64

X_test


,price
3,505


y_test


3    1
Name: target_3d, dtype: int64

In [22]:
initial_train_size = 2
test_size = 1

for start in range(
    initial_train_size,
    len(model_df),
    test_size
):
    
    train_fold = model_df.iloc[:start]
    test_fold = model_df.iloc[start:start + test_size]

    print(f"\nTrain: rows 0 to {start - 1}")
    display(train_fold)

    print(f"Test: row {start}")
    display(test_fold)


Train: rows 0 to 1


,price,price_t+3,future_return_3d,target_3d
0,500,505.0,0.01000,1
1,502,507.0,0.00996,1


Test: row 2


,price,price_t+3,future_return_3d,target_3d
2,501,504.0,0.005988,1



Train: rows 0 to 2


,price,price_t+3,future_return_3d,target_3d
0,500,505.0,0.010000,1
1,502,507.0,0.009960,1
2,501,504.0,0.005988,1


Test: row 3


,price,price_t+3,future_return_3d,target_3d
3,505,510.0,0.009901,1


In [24]:
momentum_example = pd.DataFrame({
    "price" : [500,502,501,505,507,504,510,512]
})

momentum_example["price_t-3"] = (
    momentum_example["price"].shift(3)
)

momentum_example["momentum_3d"] = (
    momentum_example["price"]
    / momentum_example["price_t-3"]
    -1
)

momentum_example

,price,price_t-3,momentum_3d
0,500,NaN,NaN
1,502,NaN,NaN
2,501,NaN,NaN
3,505,500.0,0.010000
4,507,502.0,0.009960
5,504,501.0,0.005988
6,510,505.0,0.009901
7,512,507.0,0.009862


In [25]:
# -------------------------
# MOMENTUM
# -------------------------

df["momentum_3d"] = (
    df["Adj Close"] / df["Adj Close"].shift(3) - 1
)

df["momentum_5d"] = (
    df["Adj Close"] / df["Adj Close"].shift(5) - 1
)


# -------------------------
# TREND
# -------------------------

df["ma_5"] = (
    df["Adj Close"].rolling(5).mean()
)

df["ma_20"] = (
    df["Adj Close"].rolling(20).mean()
)

df["price_vs_ma5"] = (
    df["Adj Close"] / df["ma_5"] - 1
)

df["price_vs_ma20"] = (
    df["Adj Close"] / df["ma_20"] - 1
)


# -------------------------
# VOLATILITY
# -------------------------

df["daily_return"] = (
    df["Adj Close"].pct_change()
)

df["volatility_10d"] = (
    df["daily_return"]
    .rolling(10)
    .std()
)

df["volatility_20d"] = (
    df["daily_return"]
    .rolling(20)
    .std()
)


# -------------------------
# MARKET BEHAVIOR
# -------------------------

df["daily_range"] = (
    (df["High"] - df["Low"]) / df["Close"]
)
##

NameError: name 'df' is not defined

In [26]:
# rolling(20) -> 20 gunluk hareketli pencere olusturur.
#rolling(20).mean()
#rolling(20).std()
#rolling(20).max()
rolling(20).min()

SyntaxError: invalid syntax (1261936424.py, line 1)